# ConvTran: Reproduction Study

Reproduction of *Improving Position Encoding of Transformers for Multivariate Time Series Classification*, by Foumani et al.

## Initial experiment

- Dataset: BasicMotions, official UEA train/test split.
- Official training set divided into 32 training and 8 validation samples.
- Test set: 40 samples.
- Model: official ConvTran implementation, without architecture changes.
- Epochs: 100.
- Batch size: 16.
- Seeds: 1234, 2345, 3456, 4567, 5678.
- Checkpoint selection: lowest validation loss.

In [5]:
from pathlib import Path
import subprocess
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "upstream" / "ConvTran").is_dir():
    ROOT = ROOT.parent

assert (ROOT / "upstream" / "ConvTran").is_dir(), (
    "Open the notebook from the project repository."
)

print("Repository:", ROOT)
print("Kernel Python:", sys.executable)
print(
    "Upstream commit:",
    subprocess.check_output(
        ["git", "-C", str(ROOT / "upstream" / "ConvTran"),
         "rev-parse", "HEAD"],
        text=True,
    ).strip(),
)

Repository: /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction
Kernel Python: /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction/.venv/bin/python
Upstream commit: 148afb6ca549915b7e78b05e2ec3b4ba6e052341


In [6]:
import re
import pandas as pd

pattern = re.compile(
    r"Best Model Test Summary:.*?loss:\s*([0-9.eE+-]+)"
    r".*?accuracy:\s*([0-9.eE+-]+)"
)

rows = []
for seed in [1234, 2345, 3456, 4567, 5678]:
    log = (
        ROOT / "outputs" / "baseline" / "basicmotions"
        / f"seed{seed}" / "run.log"
    )
    matches = pattern.findall(log.read_text(errors="replace"))
    assert matches, f"No final result found in {log}"

    loss, accuracy = matches[-1]
    rows.append({
        "dataset": "BasicMotions",
        "seed": seed,
        "epochs": 100,
        "batch_size": 16,
        "accuracy": float(accuracy),
        "test_loss": float(loss),
    })

runs = pd.DataFrame(rows)
runs

,dataset,seed,epochs,batch_size,accuracy,test_loss
0,BasicMotions,1234,100,16,1.0,0.161723
1,BasicMotions,2345,100,16,1.0,0.183982
2,BasicMotions,3456,100,16,1.0,0.210695
3,BasicMotions,4567,100,16,1.0,0.159011
4,BasicMotions,5678,100,16,1.0,0.182155


In [8]:
summary = runs.groupby("dataset").agg(
    runs=("seed", "count"),
    accuracy_mean=("accuracy", "mean"),
    accuracy_std=("accuracy", "std"),
)

summary

,runs,accuracy_mean,accuracy_std
dataset,,,
BasicMotions,5,1.0,0.0


## Initial findings

All five runs achieved 100% test accuracy on BasicMotions, with a sample standard deviation of 0 percentage points.

The runs share the same training/validation split. The upstream loader fixes its split random state to 1234 and reuses cached preprocessed data.

This result measures variation across training seeds on one small dataset. It does not establish reproducibility across the full benchmark.